In [1]:
!pip install catboost



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import importlib.util

print("CatBoost installed:", importlib.util.find_spec("catboost") is not None)
print("scikit-learn installed:", importlib.util.find_spec("sklearn") is not None)

CatBoost installed: True
scikit-learn installed: True


In [3]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Verify CatBoost installation
if importlib.util.find_spec("catboost") is None:
    raise ImportError(
        "CatBoost is not installed in this notebook's Python environment."
    )

from catboost import CatBoostRegressor

print("Libraries imported successfully.")

# Locate the existing processed regression dataset
possible_paths = [
    Path("../data/processed/CarbonOpt_XGBoost_Regression.xlsx"),
    Path("data/processed/CarbonOpt_XGBoost_Regression.xlsx"),
    Path.cwd().parent / "data/processed/CarbonOpt_XGBoost_Regression.xlsx",
    Path.cwd() / "data/processed/CarbonOpt_XGBoost_Regression.xlsx",
]

dataset_path = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

if dataset_path is None:
    raise FileNotFoundError(
        "Regression dataset not found. Check your CarbonOpt project folder."
    )

df = pd.read_excel(dataset_path)

print("Dataset:", dataset_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Libraries imported successfully.
Dataset: C:\Users\angel\OneDrive\Desktop\carbonOPT\data\processed\CarbonOpt_XGBoost_Regression.xlsx
Shape: (3162, 11)
Columns: ['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_lag_1', 'co2_lag_2', 'energy_growth_rate', 'co2_growth_rate', 'co2_next_year']


,state,year,residential_billion_btu,commercial_billion_btu,industrial_billion_btu,transportation_billion_btu,co2_lag_1,co2_lag_2,energy_growth_rate,co2_growth_rate,co2_next_year
0,AK,1962,10472,7129,24859,34182,4.846,4.040,9.457298,9.925712,5.366
1,AK,1963,11256,9246,25598,32387,5.327,4.846,2.408601,0.732119,5.620
2,AK,1964,13004,12021,25522,32246,5.366,5.327,5.484915,4.733507,5.720
3,AK,1965,13857,14210,22874,34379,5.620,5.366,3.050983,1.779359,6.812
4,AK,1966,15563,15066,33619,36233,5.720,5.620,17.770954,19.090909,7.801


In [4]:
target_column = "co2_next_year"

if target_column not in df.columns:
    raise ValueError(
        f"Target '{target_column}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Missing values per column:")
display(df.isnull().sum().to_frame("Missing values"))

print("Duplicate rows:", df.duplicated().sum())
print("Target summary:")
display(df[target_column].describe().to_frame())

Missing values per column:


,Missing values
state,0
year,0
residential_billion_btu,0
commercial_billion_btu,0
industrial_billion_btu,0
transportation_billion_btu,0
co2_lag_1,0
co2_lag_2,0
energy_growth_rate,0
co2_growth_rate,0


Duplicate rows: 0
Target summary:


,co2_next_year
count,3162.000000
mean,97.083362
std,101.103405
min,2.395000
25%,31.789000
50%,68.262000
75%,119.743000
max,684.268000


In [5]:
# Step 3: Prepare features and chronological split

target_column = "co2_next_year"

# Sort chronologically
df = df.sort_values(["year", "state"]).reset_index(drop=True)

# Separate input features and target
X = df.drop(columns=[target_column])
y = df[target_column]

# Use the existing chronological boundary if available
if df["year"].min() <= 2011 and df["year"].max() >= 2012:
    train_mask = df["year"] <= 2011
    test_mask = df["year"] >= 2012
else:
    split_year = sorted(df["year"].unique())[
        int(len(df["year"].unique()) * 0.8)
    ]
    train_mask = df["year"] < split_year
    test_mask = df["year"] >= split_year

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# CatBoost treats state as a categorical feature
X_train["state"] = X_train["state"].astype(str)
X_test["state"] = X_test["state"].astype(str)

cat_features = ["state"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training years:", X_train["year"].min(), "-", X_train["year"].max())
print("Testing years:", X_test["year"].min(), "-", X_test["year"].max())
print("Features:", X_train.columns.tolist())

Training rows: 2550
Testing rows: 612
Training years: 1962 - 2011
Testing years: 2012 - 2023
Features: ['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_lag_1', 'co2_lag_2', 'energy_growth_rate', 'co2_growth_rate']


In [6]:
# Step 4: Train CatBoost Regression baseline

catboost_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

# Generate predictions on the unseen test data
catboost_predictions = catboost_model.predict(X_test)

# Evaluate baseline performance
catboost_r2 = r2_score(y_test, catboost_predictions)
catboost_mae = mean_absolute_error(y_test, catboost_predictions)
catboost_rmse = np.sqrt(
    mean_squared_error(y_test, catboost_predictions)
)

print("CATBOOST BASELINE RESULTS")
print("-" * 35)
print(f"R² Score : {catboost_r2:.4f}")
print(f"MAE      : {catboost_mae:.4f}")
print(f"RMSE     : {catboost_rmse:.4f}")

CATBOOST BASELINE RESULTS
-----------------------------------
R² Score : 0.9956
MAE      : 4.2710
RMSE     : 6.9458


In [7]:
# Step 5: Overfitting diagnostic

train_predictions = catboost_model.predict(X_train)

train_r2 = r2_score(y_train, train_predictions)
train_mae = mean_absolute_error(y_train, train_predictions)
train_rmse = np.sqrt(
    mean_squared_error(y_train, train_predictions)
)

print("TRAINING PERFORMANCE")
print("-" * 35)
print(f"R² Score : {train_r2:.4f}")
print(f"MAE      : {train_mae:.4f}")
print(f"RMSE     : {train_rmse:.4f}")

print("\nTEST PERFORMANCE")
print("-" * 35)
print(f"R² Score : {catboost_r2:.4f}")
print(f"MAE      : {catboost_mae:.4f}")
print(f"RMSE     : {catboost_rmse:.4f}")

print("\nGENERALIZATION GAP")
print("-" * 35)
print(f"R² gap   : {train_r2 - catboost_r2:.4f}")
print(f"MAE gap  : {catboost_mae - train_mae:.4f}")
print(f"RMSE gap : {catboost_rmse - train_rmse:.4f}")

TRAINING PERFORMANCE
-----------------------------------
R² Score : 0.9989
MAE      : 2.3295
RMSE     : 3.2975

TEST PERFORMANCE
-----------------------------------
R² Score : 0.9956
MAE      : 4.2710
RMSE     : 6.9458

GENERALIZATION GAP
-----------------------------------
R² gap   : 0.0033
MAE gap  : 1.9415
RMSE gap : 3.6483


In [8]:
# Step 6: Generate chronological out-of-fold residuals

from catboost import CatBoostRegressor

# Use only the training period to generate residual-training data
training_years = np.sort(X_train["year"].unique())

# Reserve the earliest 50% of years as the initial training window
initial_size = max(5, int(len(training_years) * 0.5))
validation_years = training_years[initial_size:]

# Divide the remaining years into four chronological blocks
validation_blocks = [
    block for block in np.array_split(validation_years, 4)
    if len(block) > 0
]

oof_residuals = []
oof_features = []

for fold, block in enumerate(validation_blocks, start=1):
    block_start = block.min()

    fold_train_mask = X_train["year"] < block_start
    fold_valid_mask = X_train["year"].isin(block)

    X_fold_train = X_train.loc[fold_train_mask]
    y_fold_train = y_train.loc[fold_train_mask]

    X_fold_valid = X_train.loc[fold_valid_mask]
    y_fold_valid = y_train.loc[fold_valid_mask]

    fold_model = CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features
    )

    fold_predictions = fold_model.predict(X_fold_valid)

    # Residual = actual value - CatBoost prediction
    residuals = y_fold_valid.to_numpy() - fold_predictions

    oof_features.append(X_fold_valid.copy())
    oof_residuals.append(
        pd.Series(residuals, index=X_fold_valid.index)
    )

    print(
        f"Fold {fold}: train through {X_fold_train['year'].max()}, "
        f"validate {block.min()}–{block.max()}, "
        f"rows = {len(X_fold_valid)}"
    )

# Combine out-of-fold residuals and their corresponding features
X_residual_train = pd.concat(oof_features).sort_index()
y_residual_train = pd.concat(oof_residuals).sort_index()

print("\nResidual-training rows:", len(X_residual_train))
print("Residual mean:", round(y_residual_train.mean(), 4))
print("Residual standard deviation:", round(y_residual_train.std(), 4))

Fold 1: train through 1986, validate 1987–1993, rows = 357
Fold 2: train through 1993, validate 1994–1999, rows = 306
Fold 3: train through 1999, validate 2000–2005, rows = 306
Fold 4: train through 2005, validate 2006–2011, rows = 306

Residual-training rows: 1275
Residual mean: -0.0799
Residual standard deviation: 10.0851


In [9]:
# Step 7: Train SVR on out-of-fold residuals

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

# Separate categorical and numerical features
categorical_features = ["state"]
numerical_features = [
    col for col in X_residual_train.columns
    if col not in categorical_features
]

# Preprocessing for SVR
svr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# SVR residual correction model
svr_model = Pipeline([
    ("preprocessor", svr_preprocessor),
    ("svr", SVR(
        kernel="rbf",
        C=10.0,
        epsilon=0.1,
        gamma="scale"
    ))
])

# Train SVR using out-of-fold residuals
svr_model.fit(
    X_residual_train,
    y_residual_train
)

print("SVR residual correction model trained successfully.")
print("Residual training samples:", len(y_residual_train))
print("Numerical features:", len(numerical_features))
print("Categorical features:", categorical_features)

SVR residual correction model trained successfully.
Residual training samples: 1275
Numerical features: 9
Categorical features: ['state']


In [10]:
# Step 8: Evaluate CatBoost + SVR hybrid

# Predict residual errors for the test set
predicted_residuals = svr_model.predict(X_test)

# Hybrid prediction = CatBoost prediction + SVR correction
hybrid_predictions = catboost_predictions + predicted_residuals

def regression_metrics(y_true, predictions):
    return {
        "R2": r2_score(y_true, predictions),
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions))
    }

comparison = pd.DataFrame({
    "CatBoost Baseline": regression_metrics(
        y_test, catboost_predictions
    ),
    "CatBoost + SVR Hybrid": regression_metrics(
        y_test, hybrid_predictions
    )
}).T

display(comparison.round(4))

print("\nChange in test MAE:", round(
    mean_absolute_error(y_test, catboost_predictions)
    - mean_absolute_error(y_test, hybrid_predictions), 4
))

print("Change in test RMSE:", round(
    np.sqrt(mean_squared_error(y_test, catboost_predictions))
    - np.sqrt(mean_squared_error(y_test, hybrid_predictions)), 4
))

,R2,MAE,RMSE
CatBoost Baseline,0.9956,4.2710,6.9458
CatBoost + SVR Hybrid,0.9901,7.1328,10.4440



Change in test MAE: -2.8618
Change in test RMSE: -3.4981


In [11]:
# Inspect target and lag alignment by state and year

audit_df = df.sort_values(["state", "year"]).copy()

display(
    audit_df[
        [
            "state",
            "year",
            "co2_lag_1",
            "co2_lag_2",
            "co2_next_year"
        ]
    ].head(15)
)

print("Unique years:", audit_df["year"].nunique())
print("States:", audit_df["state"].nunique())

# Check whether each state-year row has the next consecutive year
audit_df["next_observed_year"] = (
    audit_df.groupby("state")["year"].shift(-1)
)

audit_df["year_gap"] = (
    audit_df["next_observed_year"] - audit_df["year"]
)

print(
    "\nRows where the next observation is not the following year:",
    (audit_df["year_gap"].notna() & (audit_df["year_gap"] != 1)).sum()
)

,state,year,co2_lag_1,co2_lag_2,co2_next_year
0,AK,1962,4.846,4.040,5.366
51,AK,1963,5.327,4.846,5.620
102,AK,1964,5.366,5.327,5.720
153,AK,1965,5.620,5.366,6.812
204,AK,1966,5.720,5.620,7.801
255,AK,1967,6.812,5.720,8.170
306,AK,1968,7.801,6.812,10.014
357,AK,1969,8.170,7.801,11.356
408,AK,1970,10.014,8.170,12.643
459,AK,1971,11.356,10.014,13.432


Unique years: 62
States: 51

Rows where the next observation is not the following year: 0


In [12]:
# Check whether lag features align with the target series

audit_df = df.sort_values(["state", "year"]).copy()

# If target is next year's CO2, target at year t should equal
# the next row's co2_lag_1 only if that lag column represents
# the actual CO2 value for year t+1. We test this relationship.
audit_df["next_year_lag1"] = (
    audit_df.groupby("state")["co2_lag_1"].shift(-1)
)

audit_df["target_vs_next_lag1_diff"] = (
    audit_df["co2_next_year"] - audit_df["next_year_lag1"]
)

display(
    audit_df[
        [
            "state",
            "year",
            "co2_lag_1",
            "co2_lag_2",
            "co2_next_year",
            "next_year_lag1",
            "target_vs_next_lag1_diff"
        ]
    ].head(15).round(4)
)

print(
    "Mean absolute difference:",
    audit_df["target_vs_next_lag1_diff"].abs().mean().round(4)
)

print(
    "Exact matches:",
    (audit_df["target_vs_next_lag1_diff"].abs() < 1e-6).sum()
)

,state,year,co2_lag_1,co2_lag_2,co2_next_year,next_year_lag1,target_vs_next_lag1_diff
0,AK,1962,4.846,4.040,5.366,5.327,0.039
51,AK,1963,5.327,4.846,5.620,5.366,0.254
102,AK,1964,5.366,5.327,5.720,5.620,0.100
153,AK,1965,5.620,5.366,6.812,5.720,1.092
204,AK,1966,5.720,5.620,7.801,6.812,0.989
255,AK,1967,6.812,5.720,8.170,7.801,0.369
306,AK,1968,7.801,6.812,10.014,8.170,1.844
357,AK,1969,8.170,7.801,11.356,10.014,1.342
408,AK,1970,10.014,8.170,12.643,11.356,1.287
459,AK,1971,11.356,10.014,13.432,12.643,0.789


Mean absolute difference: 3.6009
Exact matches: 3


In [13]:
# Inspect how target and lag values relate within each state

audit_df = df.sort_values(["state", "year"]).copy()

for col in ["co2_lag_1", "co2_lag_2", "co2_next_year"]:
    print(f"\n{col}:")
    print(audit_df.groupby("year")[col].mean().head(8).round(4))

print("\nCorrelation matrix:")
display(
    audit_df[
        [
            "co2_lag_1",
            "co2_lag_2",
            "co2_next_year"
        ]
    ].corr().round(4)
)


co2_lag_1:
year
1962    57.7194
1963    60.1125
1964    62.4873
1965    65.1387
1966    68.0281
1967    71.7549
1968    73.6120
1969    77.7993
Name: co2_lag_1, dtype: float64

co2_lag_2:
year
1962    57.1625
1963    57.7194
1964    60.1125
1965    62.4873
1966    65.1387
1967    68.0281
1968    71.7549
1969    73.6120
Name: co2_lag_2, dtype: float64

co2_next_year:
year
1962    62.4873
1963    65.1387
1964    68.0281
1965    71.7549
1966    73.6120
1967    77.7993
1968    81.1228
1969    83.5381
Name: co2_next_year, dtype: float64

Correlation matrix:


,co2_lag_1,co2_lag_2,co2_next_year
co2_lag_1,1.0000,0.9983,0.9964
co2_lag_2,0.9983,1.0000,0.9946
co2_next_year,0.9964,0.9946,1.0000


In [14]:
# Inspect the target and lag values for selected states and years

audit_df = df.sort_values(["state", "year"]).copy()

for state_name in ["AK", "CA", "TX"]:
    state_data = audit_df[
        audit_df["state"] == state_name
    ][
        [
            "state",
            "year",
            "co2_lag_1",
            "co2_lag_2",
            "co2_next_year"
        ]
    ].head(10)

    print(f"\nState: {state_name}")
    display(state_data.round(4))

# Check whether lag_1 and lag_2 are shifted versions of each other
audit_df["previous_lag1"] = (
    audit_df.groupby("state")["co2_lag_1"].shift(1)
)

print(
    "\nMean absolute difference between lag_2 and previous lag_1:",
    (audit_df["co2_lag_2"] - audit_df["previous_lag1"])
    .abs()
    .mean()
    .round(6)
)


State: AK


,state,year,co2_lag_1,co2_lag_2,co2_next_year
0,AK,1962,4.846,4.040,5.366
51,AK,1963,5.327,4.846,5.620
102,AK,1964,5.366,5.327,5.720
153,AK,1965,5.620,5.366,6.812
204,AK,1966,5.720,5.620,7.801
255,AK,1967,6.812,5.720,8.170
306,AK,1968,7.801,6.812,10.014
357,AK,1969,8.170,7.801,11.356
408,AK,1970,10.014,8.170,12.643
459,AK,1971,11.356,10.014,13.432



State: CA


,state,year,co2_lag_1,co2_lag_2,co2_next_year
4,CA,1962,208.499,197.471,219.449
55,CA,1963,209.475,208.499,238.269
106,CA,1964,219.449,209.475,243.670
157,CA,1965,238.269,219.449,260.788
208,CA,1966,243.670,238.269,266.894
259,CA,1967,260.788,243.670,284.836
310,CA,1968,266.894,260.788,288.097
361,CA,1969,284.836,266.894,294.575
412,CA,1970,288.097,284.836,306.022
463,CA,1971,294.575,288.097,312.981



State: TX


,state,year,co2_lag_1,co2_lag_2,co2_next_year
43,TX,1962,241.076,240.508,263.588
94,TX,1963,252.050,241.076,270.107
145,TX,1964,263.588,252.050,279.282
196,TX,1965,270.107,263.588,295.064
247,TX,1966,279.282,270.107,311.281
298,TX,1967,295.064,279.282,330.316
349,TX,1968,311.281,295.064,350.353
400,TX,1969,330.316,311.281,363.388
451,TX,1970,350.353,330.316,380.380
502,TX,1971,363.388,350.353,403.488



Mean absolute difference between lag_2 and previous lag_1: 0.0


In [15]:
# Check target alignment for each state

audit_df = df.sort_values(["state", "year"]).copy()

audit_df["next_year_lag1"] = (
    audit_df.groupby("state")["co2_lag_1"].shift(-1)
)

audit_df["next_year_lag2"] = (
    audit_df.groupby("state")["co2_lag_2"].shift(-1)
)

audit_df["previous_year_target"] = (
    audit_df.groupby("state")["co2_next_year"].shift(1)
)

display(
    audit_df.loc[
        audit_df["state"].eq("AK"),
        [
            "year",
            "co2_lag_1",
            "co2_lag_2",
            "co2_next_year",
            "next_year_lag1",
            "next_year_lag2",
            "previous_year_target"
        ]
    ].head(10).round(4)
)

,year,co2_lag_1,co2_lag_2,co2_next_year,next_year_lag1,next_year_lag2,previous_year_target
0,1962,4.846,4.040,5.366,5.327,4.846,NaN
51,1963,5.327,4.846,5.620,5.366,5.327,5.366
102,1964,5.366,5.327,5.720,5.620,5.366,5.620
153,1965,5.620,5.366,6.812,5.720,5.620,5.720
204,1966,5.720,5.620,7.801,6.812,5.720,6.812
255,1967,6.812,5.720,8.170,7.801,6.812,7.801
306,1968,7.801,6.812,10.014,8.170,7.801,8.170
357,1969,8.170,7.801,11.356,10.014,8.170,10.014
408,1970,10.014,8.170,12.643,11.356,10.014,11.356
459,1971,11.356,10.014,13.432,12.643,11.356,12.643


In [16]:
# Check exact target/lag relationships across the full dataset

audit_df = df.sort_values(["state", "year"]).copy()

audit_df["next_lag1"] = (
    audit_df.groupby("state")["co2_lag_1"].shift(-1)
)

audit_df["next_lag2"] = (
    audit_df.groupby("state")["co2_lag_2"].shift(-1)
)

audit_df["previous_target"] = (
    audit_df.groupby("state")["co2_next_year"].shift(1)
)

checks = {
    "target vs next-year lag_1": (
        audit_df["co2_next_year"] - audit_df["next_lag1"]
    ).abs().mean(),
    "target vs next-year lag_2": (
        audit_df["co2_next_year"] - audit_df["next_lag2"]
    ).abs().mean(),
    "target vs previous-row target": (
        audit_df["co2_next_year"] - audit_df["previous_target"]
    ).abs().mean(),
}

for name, value in checks.items():
    print(f"{name}: {value:.6f}")

print("\nDataset columns:")
print(df.columns.tolist())

target vs next-year lag_1: 3.600895
target vs next-year lag_2: 5.235386
target vs previous-row target: 3.589729

Dataset columns:
['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_lag_1', 'co2_lag_2', 'energy_growth_rate', 'co2_growth_rate', 'co2_next_year']


In [17]:
# Step 4: Train CatBoost baseline

catboost_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

# Predict on the chronological test set
catboost_predictions = catboost_model.predict(X_test)

# Evaluate
catboost_r2 = r2_score(y_test, catboost_predictions)
catboost_mae = mean_absolute_error(y_test, catboost_predictions)
catboost_rmse = np.sqrt(mean_squared_error(y_test, catboost_predictions))

print("CatBoost Baseline Results")
print("-" * 35)
print(f"R²   : {catboost_r2:.4f}")
print(f"MAE  : {catboost_mae:.4f}")
print(f"RMSE : {catboost_rmse:.4f}")

CatBoost Baseline Results
-----------------------------------
R²   : 0.9956
MAE  : 4.2710
RMSE : 6.9458


In [18]:
# Inspect lag features and target by state and year

cols = [
    "state",
    "year",
    "co2_lag_1",
    "co2_lag_2",
    "co2_next_year"
]

display(
    df[cols]
    .sort_values(["state", "year"])
    .head(20)
)

print("Unique states:", df["state"].nunique())
print("Year range:", df["year"].min(), "-", df["year"].max())

print("\nCorrelation with target:")
display(
    df[["co2_lag_1", "co2_lag_2", "co2_next_year"]]
    .corr()["co2_next_year"]
    .to_frame("Correlation")
)

,state,year,co2_lag_1,co2_lag_2,co2_next_year
0,AK,1962,4.846,4.040,5.366
51,AK,1963,5.327,4.846,5.620
102,AK,1964,5.366,5.327,5.720
153,AK,1965,5.620,5.366,6.812
204,AK,1966,5.720,5.620,7.801
255,AK,1967,6.812,5.720,8.170
306,AK,1968,7.801,6.812,10.014
357,AK,1969,8.170,7.801,11.356
408,AK,1970,10.014,8.170,12.643
459,AK,1971,11.356,10.014,13.432


Unique states: 51
Year range: 1962 - 2023

Correlation with target:


,Correlation
co2_lag_1,0.996371
co2_lag_2,0.994625
co2_next_year,1.000000


In [19]:
# Check whether the target is simply shifted CO2 data

display(
    df.groupby("state")[
        ["year", "co2_lag_1", "co2_lag_2", "co2_next_year"]
    ].head(5)
)

print("Target vs lag 1:")
print(
    (df["co2_next_year"] - df["co2_lag_1"])
    .abs()
    .describe()
)

print("\nTarget vs lag 2:")
print(
    (df["co2_next_year"] - df["co2_lag_2"])
    .abs()
    .describe()
)

,year,co2_lag_1,co2_lag_2,co2_next_year
0,1962,4.846,4.040,5.366
1,1962,61.135,62.359,69.121
2,1962,21.848,21.284,24.756
3,1962,16.913,15.589,19.324
4,1962,208.499,197.471,219.449
...,...,...,...,...
250,1966,4.416,3.737,4.701
251,1966,36.082,35.009,39.580
252,1966,71.593,68.869,78.679
253,1966,62.107,60.577,68.231


Target vs lag 1:
count    3162.000000
mean        5.221279
std         6.946404
min         0.003000
25%         1.072500
50%         2.927500
75%         6.561250
max        69.500000
dtype: float64

Target vs lag 2:
count    3162.000000
mean        6.456958
std         8.455343
min         0.000000
25%         1.435500
50%         3.724000
75%         8.027750
max        96.097000
dtype: float64


In [20]:
# Verify that lag features align with the previous years
check = df.sort_values(["state", "year"]).copy()

check["expected_lag_1"] = (
    check.groupby("state")["co2_lag_1"].shift(1)
)
check["expected_lag_2"] = (
    check.groupby("state")["co2_lag_1"].shift(2)
)

display(
    check.loc[
        check["state"] == check["state"].iloc[0],
        ["state", "year", "co2_lag_1", "co2_lag_2", "co2_next_year"]
    ].head(10)
)

print(
    "Lag 1 consistency:",
    np.isclose(
        check["co2_lag_1"],
        check["expected_lag_1"],
        equal_nan=True
    ).mean()
)

print(
    "Lag 2 consistency:",
    np.isclose(
        check["co2_lag_2"],
        check["expected_lag_2"],
        equal_nan=True
    ).mean()
)

,state,year,co2_lag_1,co2_lag_2,co2_next_year
0,AK,1962,4.846,4.040,5.366
51,AK,1963,5.327,4.846,5.620
102,AK,1964,5.366,5.327,5.720
153,AK,1965,5.620,5.366,6.812
204,AK,1966,5.720,5.620,7.801
255,AK,1967,6.812,5.720,8.170
306,AK,1968,7.801,6.812,10.014
357,AK,1969,8.170,7.801,11.356
408,AK,1970,10.014,8.170,12.643
459,AK,1971,11.356,10.014,13.432


Lag 1 consistency: 0.0009487666034155598
Lag 2 consistency: 0.0009487666034155598


In [21]:
check = df.sort_values(["state", "year"]).copy()

check["next_year_target_shift"] = (
    check.groupby("state")["co2_next_year"].shift(-1)
)

display(
    check.loc[
        check["state"] == "AK",
        ["year", "co2_lag_1", "co2_lag_2",
         "co2_next_year", "next_year_target_shift"]
    ].head(10)
)

,year,co2_lag_1,co2_lag_2,co2_next_year,next_year_target_shift
0,1962,4.846,4.040,5.366,5.620
51,1963,5.327,4.846,5.620,5.720
102,1964,5.366,5.327,5.720,6.812
153,1965,5.620,5.366,6.812,7.801
204,1966,5.720,5.620,7.801,8.170
255,1967,6.812,5.720,8.170,10.014
306,1968,7.801,6.812,10.014,11.356
357,1969,8.170,7.801,11.356,12.643
408,1970,10.014,8.170,12.643,13.432
459,1971,11.356,10.014,13.432,12.505


In [22]:
check = df.sort_values(["state", "year"]).copy()

for col in ["co2_lag_1", "co2_lag_2"]:
    check[f"{col}_next_row"] = (
        check.groupby("state")[col].shift(-1)
    )

display(
    check.loc[
        check["state"] == "AK",
        ["year", "co2_lag_1", "co2_lag_2",
         "co2_next_year", "co2_lag_1_next_row",
         "co2_lag_2_next_row"]
    ].head(10)
)


,year,co2_lag_1,co2_lag_2,co2_next_year,co2_lag_1_next_row,co2_lag_2_next_row
0,1962,4.846,4.040,5.366,5.327,4.846
51,1963,5.327,4.846,5.620,5.366,5.327
102,1964,5.366,5.327,5.720,5.620,5.366
153,1965,5.620,5.366,6.812,5.720,5.620
204,1966,5.720,5.620,7.801,6.812,5.720
255,1967,6.812,5.720,8.170,7.801,6.812
306,1968,7.801,6.812,10.014,8.170,7.801
357,1969,8.170,7.801,11.356,10.014,8.170
408,1970,10.014,8.170,12.643,11.356,10.014
459,1971,11.356,10.014,13.432,12.643,11.356


In [23]:
# Reduced-feature CatBoost experiment
# Exclude the two strongest lag features

drop_features = ["co2_lag_1", "co2_lag_2"]

X_train_reduced = X_train.drop(columns=drop_features)
X_test_reduced = X_test.drop(columns=drop_features)

reduced_cat_features = ["state"]

reduced_model = CatBoostRegressor(
    iterations=300,
    depth=4,
    learning_rate=0.03,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

reduced_model.fit(
    X_train_reduced,
    y_train,
    cat_features=reduced_cat_features
)

reduced_predictions = reduced_model.predict(X_test_reduced)

reduced_r2 = r2_score(y_test, reduced_predictions)
reduced_mae = mean_absolute_error(y_test, reduced_predictions)
reduced_rmse = np.sqrt(
    mean_squared_error(y_test, reduced_predictions)
)

print("Reduced-Feature CatBoost Results")
print("-" * 40)
print(f"R²   : {reduced_r2:.4f}")
print(f"MAE  : {reduced_mae:.4f}")
print(f"RMSE : {reduced_rmse:.4f}")

Reduced-Feature CatBoost Results
----------------------------------------
R²   : 0.9812
MAE  : 10.6102
RMSE : 14.3910


In [24]:
# Simpler CatBoost: stronger regularization and fewer features

selected_features = [
    "state",
    "year",
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu"
]

X_train_simple = X_train[selected_features].copy()
X_test_simple = X_test[selected_features].copy()

simple_model = CatBoostRegressor(
    iterations=200,
    depth=2,
    learning_rate=0.02,
    l2_leaf_reg=20,
    random_seed=42,
    loss_function="RMSE",
    verbose=False,
    allow_writing_files=False
)

simple_model.fit(
    X_train_simple,
    y_train,
    cat_features=["state"]
)

simple_predictions = simple_model.predict(X_test_simple)

print("Simplified CatBoost Results")
print("-" * 35)
print(f"R²   : {r2_score(y_test, simple_predictions):.4f}")
print(f"MAE  : {mean_absolute_error(y_test, simple_predictions):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, simple_predictions)):.4f}")

Simplified CatBoost Results
-----------------------------------
R²   : 0.9470
MAE  : 16.7425
RMSE : 24.1547


In [25]:
# Step 6: Generate chronological residuals for the simplified model

years = np.sort(X_train_simple["year"].unique())

warmup_count = max(5, int(len(years) * 0.5))
remaining_years = years[warmup_count:]
validation_blocks = np.array_split(remaining_years, 4)

oof_indices = []
oof_residuals = []

for fold, validation_years in enumerate(validation_blocks, start=1):
    if len(validation_years) == 0:
        continue

    train_mask = X_train_simple["year"] < validation_years.min()
    valid_mask = X_train_simple["year"].isin(validation_years)

    X_fold_train = X_train_simple.loc[train_mask]
    y_fold_train = y_train.loc[X_fold_train.index]

    X_fold_valid = X_train_simple.loc[valid_mask]
    y_fold_valid = y_train.loc[X_fold_valid.index]

    fold_model = CatBoostRegressor(
        iterations=200,
        depth=2,
        learning_rate=0.02,
        l2_leaf_reg=20,
        random_seed=42,
        loss_function="RMSE",
        verbose=False,
        allow_writing_files=False
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=["state"]
    )

    fold_pred = fold_model.predict(X_fold_valid)

    oof_indices.extend(X_fold_valid.index.tolist())
    oof_residuals.extend(
        (y_fold_valid.to_numpy() - fold_pred).tolist()
    )

    print(
        f"Fold {fold}: trained through {X_fold_train['year'].max()}, "
        f"validated {validation_years.min()}–{validation_years.max()}"
    )

residual_X = X_train_simple.loc[oof_indices].copy()
residual_y = np.asarray(oof_residuals)

print("\nResidual training rows:", len(residual_X))

Fold 1: trained through 1986, validated 1987–1993
Fold 2: trained through 1993, validated 1994–1999
Fold 3: trained through 1999, validated 2000–2005
Fold 4: trained through 2005, validated 2006–2011

Residual training rows: 1275


In [26]:
# Step 7: Train SVR on out-of-sample CatBoost residuals

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

numeric_features = [
    "year",
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("state", OneHotEncoder(handle_unknown="ignore"), ["state"])
    ]
)

svr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1))
])

svr_model.fit(residual_X, residual_y)

print("SVR residual corrector trained.")
print("Training samples:", len(residual_X))

SVR residual corrector trained.
Training samples: 1275


In [27]:
# Step 8: Evaluate the hybrid model

# Predict residual corrections for the test set
svr_corrections = svr_model.predict(X_test_simple)

# Hybrid prediction = CatBoost prediction + SVR correction
hybrid_predictions = simple_predictions + svr_corrections

# Calculate evaluation metrics
hybrid_r2 = r2_score(y_test, hybrid_predictions)
hybrid_mae = mean_absolute_error(y_test, hybrid_predictions)
hybrid_rmse = np.sqrt(
    mean_squared_error(y_test, hybrid_predictions)
)

print("CatBoost–SVR Hybrid Results")
print("-" * 40)
print(f"R²   : {hybrid_r2:.4f}")
print(f"MAE  : {hybrid_mae:.4f}")
print(f"RMSE : {hybrid_rmse:.4f}")

print("\nComparison with simplified CatBoost")
print("-" * 40)
print(f"CatBoost R² : {r2_score(y_test, simple_predictions):.4f}")
print(f"Hybrid R²   : {hybrid_r2:.4f}")
print(f"CatBoost MAE: {mean_absolute_error(y_test, simple_predictions):.4f}")
print(f"Hybrid MAE  : {hybrid_mae:.4f}")

CatBoost–SVR Hybrid Results
----------------------------------------
R²   : 0.9576
MAE  : 14.3220
RMSE : 21.6070

Comparison with simplified CatBoost
----------------------------------------
CatBoost R² : 0.9470
Hybrid R²   : 0.9576
CatBoost MAE: 16.7425
Hybrid MAE  : 14.3220


In [28]:
from pathlib import Path
import pandas as pd

output_dir = Path("../outputs/final")
output_dir.mkdir(parents=True, exist_ok=True)

comparison = pd.DataFrame([
    {
        "Model": "Simplified CatBoost",
        "R2": r2_score(y_test, simple_predictions),
        "MAE": mean_absolute_error(y_test, simple_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, simple_predictions))
    },
    {
        "Model": "CatBoost-SVR Hybrid",
        "R2": r2_score(y_test, hybrid_predictions),
        "MAE": mean_absolute_error(y_test, hybrid_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, hybrid_predictions))
    }
])

comparison.to_csv(
    output_dir / "catboost_svr_hybrid_comparison.csv",
    index=False
)

display(comparison.round(4))
print("Saved to:", (output_dir / "catboost_svr_hybrid_comparison.csv").resolve())

,Model,R2,MAE,RMSE
0,Simplified CatBoost,0.9470,16.7425,24.1547
1,CatBoost-SVR Hybrid,0.9576,14.3220,21.6070


Saved to: C:\Users\angel\OneDrive\Desktop\carbonOPT\outputs\final\catboost_svr_hybrid_comparison.csv


In [29]:

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

output_dir = Path("../outputs/final")
output_dir.mkdir(parents=True, exist_ok=True)

def calculate_metrics(actual, predicted):
    return {
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted))
    }

comparison_results = pd.DataFrame([
    {
        "Model": "Existing XGBoost",
        "R2": 0.8663,
        "MAE": 15.3435,
        "RMSE": 38.1241,
        "Evaluation": "Earlier experiment"
    },
    {
        "Model": "Random Forest",
        "R2": 0.933782,
        "MAE": 19.178475,
        "RMSE": 26.997305,
        "Evaluation": "Earlier experiment"
    },
    {
        "Model": "Simplified CatBoost",
        **calculate_metrics(y_test, simple_predictions),
        "Evaluation": "Current test split"
    },
    {
        "Model": "Proposed CatBoost-SVR Hybrid",
        **calculate_metrics(y_test, hybrid_predictions),
        "Evaluation": "Current test split"
    }
])

display(comparison_results.round(4))

comparison_results.to_csv(
    output_dir / "model_comparison_results.csv",
    index=False
)

comparison_results.to_excel(
    output_dir / "model_comparison_results.xlsx",
    index=False
)

print("Comparison results saved to:", output_dir.resolve())


,Model,R2,MAE,RMSE,Evaluation
0,Existing XGBoost,0.8663,15.3435,38.1241,Earlier experiment
1,Random Forest,0.9338,19.1785,26.9973,Earlier experiment
2,Simplified CatBoost,0.9470,16.7425,24.1547,Current test split
3,Proposed CatBoost-SVR Hybrid,0.9576,14.3220,21.6070,Current test split


Comparison results saved to: C:\Users\angel\OneDrive\Desktop\carbonOPT\outputs\final


In [30]:

validation_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "R2": 0.933782,
        "MAE": 19.178475,
        "RMSE": 26.997305,
        "Evaluation": "Earlier experiment"
    },
    {
        "Model": "Simplified CatBoost",
        **calculate_metrics(y_test, simple_predictions),
        "Evaluation": "Current test split"
    },
    {
        "Model": "CatBoost-SVR Hybrid",
        **calculate_metrics(y_test, hybrid_predictions),
        "Evaluation": "Current test split"
    }
])

display(validation_results.round(4))

validation_results.to_csv(
    output_dir / "hybrid_validation_results.csv",
    index=False
)

print("Validation results saved to:", output_dir.resolve())


,Model,R2,MAE,RMSE,Evaluation
0,Random Forest,0.9338,19.1785,26.9973,Earlier experiment
1,Simplified CatBoost,0.9470,16.7425,24.1547,Current test split
2,CatBoost-SVR Hybrid,0.9576,14.3220,21.6070,Current test split


Validation results saved to: C:\Users\angel\OneDrive\Desktop\carbonOPT\outputs\final


In [31]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("Original CatBoost-SVR R²:", 
      r2_score(y_test, existing_hybrid_predictions))

print("New stacking R²:", 
      r2_score(y_test, stacked_predictions))

print("Original CatBoost-SVR MAE:",
      mean_absolute_error(y_test, existing_hybrid_predictions))

print("New stacking MAE:",
      mean_absolute_error(y_test, stacked_predictions))

print("Original CatBoost-SVR RMSE:",
      np.sqrt(mean_squared_error(y_test, existing_hybrid_predictions)))

print("New stacking RMSE:",
      np.sqrt(mean_squared_error(y_test, stacked_predictions)))

NameError: name 'existing_hybrid_predictions' is not defined

In [ ]:
required_vars = [
    "y_test",
    "existing_hybrid_predictions",
    "stacked_predictions"
]

for var in required_vars:
    print(f"{var}: {'Available' if var in globals() else 'Missing'}")

y_test: Missing
existing_hybrid_predictions: Missing
stacked_predictions: Missing


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

results = {
    "CatBoost-SVR Hybrid": existing_hybrid_predictions,
    "CatBoost-SVR-LightGBM Stacking": stacked_predictions
}

for name, predictions in results.items():
    print(f"\n{name}")
    print(f"R²:   {r2_score(y_test, predictions):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, predictions):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")

NameError: name 'existing_hybrid_predictions' is not defined

In [32]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Check which prediction variable your notebook created
print("Available prediction variables:")
for name in ["y_pred", "y_pred_hybrid", "hybrid_predictions",
             "y_pred_test", "existing_hybrid_predictions"]:
    if name in globals():
        print(name, "—", len(globals()[name]))

Available prediction variables:
hybrid_predictions — 612


In [33]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("Test rows:", len(y_test))
print("Hybrid prediction rows:", len(hybrid_predictions))

assert len(y_test) == len(hybrid_predictions), \
    "Prediction and test-target lengths do not match."

print("\nCatBoost–SVR Hybrid Results")
print(f"R²:   {r2_score(y_test, hybrid_predictions):.10f}")
print(f"MAE:  {mean_absolute_error(y_test, hybrid_predictions):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, hybrid_predictions)):.4f}")

Test rows: 612
Hybrid prediction rows: 612

CatBoost–SVR Hybrid Results
R²:   0.9575846535
MAE:  14.3220
RMSE: 21.6070
